# Lección 6.3 — Boosting: aprender de los errores (AdaBoost y Gradient Boosting)

> **Módulo 6 · Ensambles: la sabiduría de la multitud** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 110–130 min (lección con ejercicios intercalados ≈ 80 min · ejercicios finales ≈ 35 min)
> 📋 Requisitos previos: Lección 6.1 (por qué combinar modelos), Lección 6.2 (bagging y Random Forest), Lección 3.3 (descenso de gradiente)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía la diferencia entre **bagging** (en paralelo) y **boosting** (en fila).
- Describir cómo **AdaBoost** da más peso a los ejemplos mal clasificados.
- Hacer **a mano** una ronda de **Gradient Boosting**: calcular residuos y corregirlos.
- Predecir el efecto de `n_estimators`, `learning_rate`, `max_depth` y `subsample`, y por qué el boosting **sí** puede sobreajustar con demasiados árboles.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (AdaBoostClassifier, GradientBoostingClassifier,
                              GradientBoostingRegressor, RandomForestClassifier)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import make_moons, make_hastie_10_2

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el golfista y el estudiante

**El golfista.** El hoyo está a 200 m. El primer golpe deja la pelota a 50 m del hoyo. El segundo golpe **no** vuelve a apuntar desde la salida: apunta a **lo que falta**. El tercero, a lo que falte después del segundo. Cada golpe **corrige el error** del anterior.

**El estudiante.** Después de un examen de práctica, un buen estudiante no repasa todo por igual: dedica más tiempo a **las preguntas que falló**.

Eso es **boosting**: entrenar modelos **en fila**, donde cada uno se concentra en **lo que los anteriores hicieron mal**. La predicción final es la **suma** (o el voto ponderado) de todos.

| | Bagging / Random Forest (6.2) | Boosting (hoy) |
|---|---|---|
| Cómo se entrenan | En **paralelo**, cada uno por su cuenta | En **fila**, cada uno mira los errores del anterior |
| Qué tipo de modelo | Árboles **profundos** (poco sesgo, mucha varianza) | Árboles **pequeños** (mucho sesgo, poca varianza) |
| Qué reduce | **Varianza** | **Sesgo** (y algo de varianza) |
| Analogía | 100 médicos que opinan sin hablar entre sí | Un equipo de relevos: cada uno arregla lo que dejó el anterior |

### ✋ Ejercicio 1 — Golpe a golpe
Un golfista muy prudente siempre golpea con fuerza para cubrir solo el **50 %** de la distancia que le falta. El hoyo está a 200 m.
1. ¿A cuántos metros del hoyo queda después de 1, 2 y 3 golpes?
2. ¿Llegará alguna vez exactamente al hoyo? ¿Y si cubriera el 100 % en cada golpe, pero con un pulso tembloroso?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. 100 m → 50 m → 25 m. Cada golpe corrige la **mitad** del error restante.
2. Nunca exactamente, pero se acerca tanto como quiera con suficientes golpes. Con el 100 % y pulso tembloroso, cada golpe **sobrecorrige** el error del anterior y la pelota puede bailar alrededor del hoyo.

Ese "50 %" tiene nombre en boosting: es la **tasa de aprendizaje** (`learning_rate`). Golpes cortos = más golpes, pero más control. Lo verás en acción en la Sección 5.
</details>

---
## 2. 📖 AdaBoost: subir el volumen a los errores

**AdaBoost** (*Adaptive Boosting*, Freund y Schapire, 1997) fue el primer boosting exitoso. Usa **tocones** (árboles de una sola pregunta, Lección 5.4) y funciona así:

1. Todos los ejemplos empiezan con el **mismo peso**.
2. Entrena un tocón que intenta acertar, dando más importancia a los ejemplos con más peso.
3. Calcula su error. Cuanto mejor es, **más fuerza de voto** recibe al final.
4. **Sube el peso** de los ejemplos que falló y **baja** el de los que acertó.
5. Repite: el siguiente tocón se ve obligado a concentrarse en los casos difíciles.
6. Predicción final: **voto ponderado** de todos los tocones.

Vamos a hacer tres rondas "a mano" con 60 puntos. El **tamaño** de cada punto es su peso.

In [ ]:
X_mini, y_mini = make_moons(n_samples=60, noise=0.25, random_state=4)
pesos = np.ones(len(y_mini)) / len(y_mini)          # 1. todos empiezan con el mismo peso

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ronda, ax in enumerate(axes, start=1):
    tocon = DecisionTreeClassifier(max_depth=1).fit(X_mini, y_mini, sample_weight=pesos)   # 2.
    fallos = tocon.predict(X_mini) != y_mini
    error = pesos[fallos].sum()                                                  # 3. error ponderado
    fuerza = 0.5 * np.log((1 - error) / error)                                   #    fuerza de su voto

    ax.scatter(*X_mini.T, c=y_mini, cmap="coolwarm", s=pesos * 3000, edgecolor="k")
    columna, corte = tocon.tree_.feature[0], tocon.tree_.threshold[0]
    (ax.axvline if columna == 0 else ax.axhline)(corte, color="green", lw=2)
    ax.set_title(f"Ronda {ronda}: error={error:.2f} · fuerza={fuerza:.2f}")
    print(f"Ronda {ronda}: el tocón pregunta por x{columna} ≤ {corte:.2f}, falla {fallos.sum()} puntos, error ponderado={error:.2f}, fuerza del voto={fuerza:.2f}")

    pesos = pesos * np.exp(np.where(fallos, fuerza, -fuerza))                    # 4. suben los fallos
    pesos = pesos / pesos.sum()
plt.tight_layout(); plt.show()

### 🔮 Ejercicio 2 — Lee las rondas
Observa los gráficos (la línea verde es la pregunta de cada tocón; el tamaño de cada punto es su peso **al empezar** esa ronda).
1. En la ronda 2, ¿qué puntos se ven más grandes? ¿Por qué?
2. ¿Por qué el tocón de la ronda 2 o 3 hace una pregunta **distinta** a la de la ronda 1?
3. Si un tocón tuviera un error ponderado de **0.5** (como lanzar una moneda), ¿cuánta fuerza de voto recibiría? (Pista: log(1) = 0.)

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Los puntos que el tocón de la ronda 1 **clasificó mal**: AdaBoost les subió el peso para que el siguiente tocón no pueda ignorarlos.
2. Porque con los nuevos pesos, la pregunta de la ronda 1 ya **no** es la mejor: los puntos que falló ahora "pesan" mucho. El tocón siguiente busca una pregunta que acierte sobre todo **esos** puntos. Así se consigue la **diversidad** de la Lección 6.1, pero dirigida: cada miembro se especializa en lo que faltaba.
3. **Cero.** Un tocón que no sabe más que una moneda no tiene voz en el voto final. Cuanto más bajo el error, más fuerza (el de la ronda 1, con error ≈0.20, recibe ≈0.69).
</details>

### 📖 ¿Pueden 400 tocones ganarle a un árbol grande?
En la Lección 6.1 viste que **promediar** tocones no arreglaba su sesgo. ¿Y si los ponemos **en fila** con AdaBoost?

Usaremos un problema clásico de boosting: 10 características y una clase que depende de si el punto está **dentro o fuera de una "esfera"** en 10 dimensiones. Un tocón, con una sola pregunta, no puede describir una esfera.

In [ ]:
X_esf, y_esf = make_hastie_10_2(n_samples=4_000, random_state=0)
y_esf = (y_esf > 0).astype(int)                   # 1 = fuera de la esfera, 0 = dentro
Xe_tr, Xe_te, ye_tr, ye_te = train_test_split(X_esf, y_esf, test_size=0.5, random_state=0)

tocon = DecisionTreeClassifier(max_depth=1).fit(Xe_tr, ye_tr)
arbol_grande = DecisionTreeClassifier(random_state=0).fit(Xe_tr, ye_tr)
print(f"Un tocón              : {tocon.score(Xe_te, ye_te):.3f}")
print(f"Un árbol sin límites  : {arbol_grande.score(Xe_te, ye_te):.3f}  ({arbol_grande.get_n_leaves()} hojas)")

### 🔮 Ejercicio 3 — La rebelión de los tocones
Un tocón solo acierta ≈54 % (casi una moneda). Un árbol sin límites, ≈74 %.

¿Qué exactitud en prueba esperas de **AdaBoost con 400 tocones**? ¿Le ganará al árbol grande?

✍️ *Mi apuesta:*

In [ ]:
for n in [1, 10, 50, 200, 400]:      # ✏️ prueba 1000 (tarda unos segundos más)
    ada = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=n, random_state=0).fit(Xe_tr, ye_tr)
    print(f"AdaBoost con {n:>4} tocones → entrenamiento={ada.score(Xe_tr, ye_tr):.3f}  prueba={ada.score(Xe_te, ye_te):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

¡**Sí**, y por mucho! Con 400 tocones, AdaBoost llega a ≈**0.89** en prueba, contra ≈0.74 del árbol grande. Con 10 tocones todavía va en ≈0.65, y con 50, ≈0.77.

Esto responde a la pregunta que dejó abierta la Lección 6.1: los modelos simples (mucho **sesgo**) **sí** sirven para ensambles, pero combinados **en fila**. Cada tocón aporta una pieza pequeña y la suma construye una frontera complejísima. **El boosting reduce el sesgo.**
</details>

---
## 3. 📖 Gradient Boosting: perseguir los residuos

AdaBoost repesa **ejemplos**. **Gradient Boosting** (Friedman, 2001) hace algo más directo, como el golfista: cada árbol nuevo intenta predecir **lo que le falta** a la predicción actual.

Para regresión, "lo que falta" es el **residuo**: `residuo = valor real − predicción actual`.

```
Paso 0: predicción = el promedio de todos (un punto de partida tonto)
Paso 1: residuos = real − predicción
        entrena un árbol PEQUEÑO para predecir los residuos
        predicción = predicción + learning_rate × (lo que dice ese árbol)
Paso 2: residuos nuevos (más pequeños) → otro árbol → otra corrección
...      repite n_estimators veces
```

### ✋ Ejercicio 4 — Una ronda a mano
Cuatro departamentos con su precio real (en millones):

| Depto | m² | Precio real |
|---|---|---|
| A | 45 | 2.0 |
| B | 60 | 3.0 |
| C | 80 | 4.0 |
| D | 150 | 7.0 |

1. **Paso 0:** ¿cuál es la predicción inicial (el promedio)? ¿Cuál es el residuo de cada depto?
2. **Paso 1:** un tocón aprende la regla *"si m² > 100, residuo = +3; si no, residuo = −1"* (el promedio de los residuos de cada grupo). Con `learning_rate = 0.5`, ¿cuál es la nueva predicción de cada depto?
3. ¿Bajaron los residuos?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Promedio = (2 + 3 + 4 + 7) / 4 = **4.0**. Residuos: A = −2, B = −1, C = 0, D = +3.
2. A, B, C: 4.0 + 0.5 × (−1) = **3.5**. D: 4.0 + 0.5 × 3 = **5.5**.
3. Nuevos residuos: A = −1.5, B = −0.5, C = +0.5, D = +1.5. Todos más pequeños o iguales en tamaño que antes (la suma de sus valores absolutos baja de 6 a 4). El próximo árbol perseguirá **estos** residuos, y así sucesivamente.

¿Por qué se llama "gradiente"? Porque los residuos son exactamente la dirección en la que hay que moverse para bajar la pérdida (el error cuadrático), como el excursionista de la Lección 3.3. Con otras pérdidas (p. ej. clasificación) se usan "pseudo-residuos", pero la idea es la misma.
</details>

Ahora el mismo algoritmo, en código, con 80 puntos ruidosos con forma de ola (como en la Lección 6.1). Fíjate en lo corto que es: **todo Gradient Boosting cabe en unas pocas líneas**.

In [ ]:
rng_ola = np.random.default_rng(0)
x = np.sort(rng_ola.uniform(0, 10, 80))
y = np.sin(x) + rng_ola.normal(0, 0.4, 80)
x_fino = np.linspace(0, 10, 500)

def boosting_a_mano(n_rondas, learning_rate=0.1, max_depth=2):
    pred = np.full(len(x), y.mean())                 # paso 0: el promedio
    pred_fino = np.full(len(x_fino), y.mean())
    for _ in range(n_rondas):
        residuos = y - pred                          # lo que falta
        arbol = DecisionTreeRegressor(max_depth=max_depth, random_state=0).fit(x[:, None], residuos)
        pred += learning_rate * arbol.predict(x[:, None])           # corregimos un poquito
        pred_fino += learning_rate * arbol.predict(x_fino[:, None])
    return pred, pred_fino

### 🔮 Ejercicio 5 — Ronda a ronda
Vamos a dibujar la predicción después de 1, 5, 20 y 100 rondas (con `learning_rate=0.1` y árboles de profundidad 2).
1. ¿Cómo será la curva después de **1** ronda?
2. ¿Qué crees que pasará con el error de **entrenamiento** a medida que sumamos rondas?

✍️ *Mis predicciones:*

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharey=True)
for ax, n in zip(axes, [1, 5, 20, 100]):
    pred, pred_fino = boosting_a_mano(n)
    ax.scatter(x, y, s=10, c="gray"); ax.plot(x_fino, np.sin(x_fino), "k--", lw=1)
    ax.plot(x_fino, pred_fino, "r", lw=2)
    mse = np.mean((y - pred) ** 2)
    ax.set_title(f"{n} ronda(s) · error de entrenamiento {mse:.3f}")
    print(f"{n:>3} ronda(s) → error cuadrático medio en entrenamiento: {mse:.3f}")
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. Casi **plana**: el promedio más una décima parte de un árbol pequeño. Una ronda apenas mueve la predicción.
2. El error de entrenamiento **baja siempre** (≈0.51 → ≈0.36 → ≈0.16 → ≈0.04). Con 20 rondas la curva ya sigue la ola; con 100 empieza a perseguir puntos sueltos. ⚠️ Que el error de entrenamiento baje sin parar **no** es buena noticia: el boosting puede llegar a memorizar. Lo verás en la Sección 5.
</details>

### ✋ Ejercicio 6 — ¿Es lo mismo que scikit-learn?
`GradientBoostingRegressor` hace exactamente lo que acabamos de programar. Ejecuta la celda: ¿qué tan distintas son sus predicciones de las nuestras?

✍️ *Mi respuesta:*

In [ ]:
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=2, random_state=0).fit(x[:, None], y)
pred_mano, _ = boosting_a_mano(100)
print(f"Diferencia máxima entre las dos versiones: {np.abs(gb.predict(x[:, None]) - pred_mano).max():.2e}")

<details><summary><b>👀 Ver respuesta</b></summary>

La diferencia es del orden de **10⁻¹⁶**: cero, salvo errores de redondeo de la computadora. No hay magia escondida: Gradient Boosting **es** "árboles pequeños persiguiendo residuos, sumados con una tasa de aprendizaje". Todo lo demás (XGBoost, LightGBM, CatBoost, Lección 6.4) son mejoras de **velocidad** y de **control** sobre esta misma idea.
</details>

---
## 4. 📖 Qué pasa si… sumamos árboles sin parar

En Random Forest, más árboles **nunca** hacían daño (Lección 6.2). En boosting es distinto: cada árbol nuevo **corrige los residuos que quedan**, y cuando ya no queda patrón, empieza a corregir el **ruido**. Es sobreajuste, y llega poco a poco.

Para verlo usamos 1,000 puntos **nuevos** de la misma ola como conjunto de prueba. `staged_predict` nos da la predicción después de cada árbol, sin reentrenar.

### 🔮 Ejercicio 7 — ¿Dónde está el mejor número de árboles?
Con `learning_rate=0.1` y 1,000 árboles, dibuja mentalmente las dos curvas de error (entrenamiento y prueba) contra el número de árboles. ¿Qué forma tiene cada una?

✍️ *Mi predicción:*

In [ ]:
rng_prueba = np.random.default_rng(1)
x_te = rng_prueba.uniform(0, 10, 1_000)
y_te = np.sin(x_te) + rng_prueba.normal(0, 0.4, 1_000)

def curvas(learning_rate, n=1_000, max_depth=2, subsample=1.0):
    gb = GradientBoostingRegressor(n_estimators=n, learning_rate=learning_rate, max_depth=max_depth,
                                   subsample=subsample, random_state=0).fit(x[:, None], y)
    err_tr = [np.mean((y - p) ** 2) for p in gb.staged_predict(x[:, None])]
    err_te = [np.mean((y_te - p) ** 2) for p in gb.staged_predict(x_te[:, None])]
    return np.array(err_tr), np.array(err_te)

err_tr, err_te = curvas(learning_rate=0.1)
plt.plot(err_tr, label="Entrenamiento"); plt.plot(err_te, label="Prueba")
plt.axvline(err_te.argmin() + 1, color="gray", ls=":", label=f"mejor: {err_te.argmin() + 1} árboles")
plt.xscale("log"); plt.xlabel("Número de árboles (escala log)"); plt.ylabel("Error cuadrático medio")
plt.legend(); plt.grid(alpha=.3); plt.show()
print(f"Mejor número de árboles: {err_te.argmin() + 1} (error de prueba {err_te.min():.3f})")
print(f"Con 1,000 árboles: entrenamiento {err_tr[-1]:.3f} · prueba {err_te[-1]:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Entrenamiento:** baja todo el tiempo, hasta casi cero. El modelo termina memorizando los 80 puntos.
- **Prueba:** baja, toca fondo y **vuelve a subir**: la "U" del sobreajuste (Lección 4.4), ahora contra el número de árboles. El mejor punto está en **≈46 árboles** (error de prueba ≈0.205); con 1,000 árboles el entrenamiento llega a ≈0.000 y la prueba empeora a ≈0.33.

Ojo: aquí elegimos el mejor número **mirando la prueba**, solo para enseñar la curva. En la práctica se usa un conjunto de **validación**: scikit-learn lo hace solo con `n_iter_no_change` (**parada temprana**, *early stopping*), que detiene el entrenamiento cuando el error de validación deja de mejorar. Lo verás a fondo en la Lección 7.2.
</details>

---
## 5. 📖 Hiperparámetros: las perillas del boosting

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `n_estimators` | Número de árboles (rondas) | Menos sesgo; con demasiados, **sobreajuste** | Subajuste | 100–1,000, **con parada temprana** |
| `learning_rate` | Cuánto corrige cada árbol (el "golpe" del golfista) | Aprende rápido pero brusco; sobreajusta antes | Necesita más árboles, pero suele generalizar mejor | 0.05–0.1 |
| `max_depth` | Tamaño de cada árbol: cuántas variables pueden **interactuar** | Árboles más complejos, más riesgo de sobreajuste | Más simple; `1` = tocones (efectos sin interacciones) | 3 (2–6) |
| `subsample` | Fracción de ejemplos (al azar) para cada árbol | — | Más azar y diversidad, regulariza (*stochastic gradient boosting*) | 0.8 (0.5–1.0) |
| `min_samples_leaf` | Ejemplos mínimos por hoja | Más simple | Más complejo | 1–20 |
| `max_features` | Columnas candidatas en cada división | — | Más azar, como en Random Forest | `None` o 0.5–0.8 |
| `n_iter_no_change` | Parada temprana: rondas sin mejora antes de parar | — | — | 10–50 (con `validation_fraction=0.1`) |

**AdaBoost** tiene menos perillas: `n_estimators`, `learning_rate` (encoge la fuerza de cada voto) y `estimator` (el modelo base, por defecto un tocón).

### 🧠 Regla mnemotécnica
> **`learning_rate` y `n_estimators` son un equipo: si bajas uno, sube el otro.** Golpes más cortos necesitan más golpes.

### 🔮 Ejercicio 8 — Golpes cortos y golpes largos
Compararemos tres tasas de aprendizaje: **1.0**, **0.1** y **0.01**, con 1,000 árboles cada una.
1. ¿Cuál llegará antes a su mejor punto?
2. ¿Cuál tendrá el mejor error de prueba en su mejor punto?
3. ¿Cuál será más "tolerante" si te equivocas con el número de árboles?

✍️ *Mis predicciones:*

In [ ]:
for lr in [1.0, 0.1, 0.01]:      # ✏️
    err_tr, err_te = curvas(learning_rate=lr)
    plt.plot(np.arange(1, 1_001), err_te, label=f"learning_rate={lr}")
    print(f"learning_rate={lr:<5} mejor con {err_te.argmin() + 1:>4} árboles → error de prueba {err_te.min():.3f} · "
          f"con 1,000 árboles: {err_te[-1]:.3f}")
plt.xscale("log"); plt.xlabel("Número de árboles"); plt.ylabel("Error de prueba")
plt.legend(); plt.grid(alpha=.3); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Con estos datos sale (aprox.):

| `learning_rate` | Mejor nº de árboles | Mejor error de prueba | Error con 1,000 |
|---|---|---|---|
| 1.0 | 3 | ≈0.23 | ≈0.33 |
| 0.1 | 46 | ≈0.21 | ≈0.33 |
| 0.01 | ≈500 | ≈0.21 | ≈0.22 |

1. **1.0** llega a su mejor punto en un par de árboles… y empieza a sobreajustar casi de inmediato.
2. **0.1** y **0.01** llegan a un error parecido (≈0.21) y mejor que 1.0: los golpes cortos permiten un ajuste más fino.
3. **0.01** es la más tolerante: su "valle" es ancho, así que equivocarte por cientos de árboles casi no duele. El precio: **10 veces más árboles** (más tiempo).

Receta habitual: `learning_rate` pequeño (0.05–0.1) + muchos árboles + **parada temprana**.
</details>

### ✏️ Ejercicio 9 — Tamaño de cada árbol (`max_depth`)
Ahora fijamos `learning_rate=0.1` y 100 árboles, y cambiamos la profundidad de **cada** árbol. Antes de ejecutar: ¿qué profundidad crees que sobreajustará más?

In [ ]:
for d in [1, 2, 4, 8]:          # ✏️
    gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=d, random_state=0).fit(x[:, None], y)
    print(f"max_depth={d}  entrenamiento={np.mean((y - gb.predict(x[:, None])) ** 2):.3f}  "
          f"prueba={np.mean((y_te - gb.predict(x_te[:, None])) ** 2):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

`max_depth=8` sobreajusta más: entrenamiento **≈0.000** (memoriza los 80 puntos) y la peor prueba (≈0.33). La mejor prueba es la de `max_depth=2` (≈0.22); los tocones (`max_depth=1`) se quedan algo cortos (≈0.25) y con 4 ya se nota el sobreajuste (≈0.30). En boosting, **cada árbol debe ser débil**: la complejidad la da la **suma**, no cada árbol.

💡 En datos con muchas columnas, `max_depth` también decide cuántas variables pueden **combinarse** en una regla: con `max_depth=1` cada árbol mira una sola variable (efectos que solo se suman); con 3, hasta tres variables a la vez (interacciones). Lo comprobarás en la práctica integradora.
</details>

### ✏️ Ejercicio 10 — Un poco de azar (`subsample`)
Con `subsample=0.5`, cada árbol se entrena con la **mitad** de los ejemplos, elegidos al azar en cada ronda (es la idea del bagging metida dentro del boosting). Compara las curvas de prueba con `subsample=1.0` y `0.5`.

In [ ]:
for ss in [1.0, 0.5]:            # ✏️ prueba 0.8 y 0.3
    err_tr, err_te = curvas(learning_rate=0.1, subsample=ss)
    plt.plot(np.arange(1, 1_001), err_te, label=f"subsample={ss}")
    print(f"subsample={ss}: mejor error de prueba {err_te.min():.3f} (con {err_te.argmin() + 1} árboles) · con 1,000 árboles: {err_te[-1]:.3f}")
plt.xscale("log"); plt.xlabel("Número de árboles"); plt.ylabel("Error de prueba"); plt.legend(); plt.grid(alpha=.3); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Con `subsample=0.5` el mejor error de prueba mejora un poco (≈0.197 contra ≈0.205) y con 1,000 árboles sobreajusta un poco menos (≈0.319 contra ≈0.326). El efecto es modesto en estos datos, pero va siempre en la misma dirección: el azar hace que cada árbol vea un ruido distinto y le cuesta más memorizar. Además, cada árbol se entrena con la mitad de los datos: **más rápido**. Por eso 0.8 es un valor de partida muy común (en XGBoost y LightGBM se llama igual, `subsample`, Lección 6.4).
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Entrenamiento ≈ perfecto, validación empeora al sumar árboles | Demasiados árboles para ese `learning_rate` | Parada temprana (`n_iter_no_change`), bajar `learning_rate` |
| "Pongo 5,000 árboles como en Random Forest, total no sobreajusta" | Confundir bagging con boosting | En boosting el número de árboles **sí** se ajusta |
| Tarda muchísimo en entrenar | Los árboles se entrenan **en fila**: no se paraleliza como un bosque | `HistGradientBoosting`, XGBoost o LightGBM (Lección 6.4) |
| Muy buen resultado con `learning_rate=1.0` en una prueba y malo en otra | Golpes demasiado bruscos: modelo inestable | `learning_rate` 0.05–0.1 |
| Rinde mal con etiquetas muy ruidosas o atípicos extremos | El boosting insiste en los casos que falla, incluidos los **errores de etiqueta** (AdaBoost es el más sensible) | Limpiar datos, `subsample` < 1, `min_samples_leaf` más alto, pérdida robusta (`loss="huber"` en regresión) |
| Árboles profundos (`max_depth=10`) y sobreajuste rápido | Cada árbol ya es fuerte; la suma memoriza | `max_depth` 2–4 |

### 🔮 Ejercicio 11 — ¿Quién gana en la esfera?
En el problema de la esfera (Sección 2), ¿quién crees que gana: **Random Forest** (300 árboles profundos en paralelo) o **Gradient Boosting** (300 árboles de profundidad 2 en fila, `learning_rate=0.1`)?

✍️ *Mi apuesta:*

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(Xe_tr, ye_tr)
gbc = GradientBoostingClassifier(n_estimators=300, learning_rate=0.1, max_depth=2, random_state=0).fit(Xe_tr, ye_tr)
print(f"Random Forest     : {rf.score(Xe_te, ye_te):.3f}")
print(f"Gradient Boosting : {gbc.score(Xe_te, ye_te):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Gana **Gradient Boosting**: ≈0.91 contra ≈0.87. La esfera es una suma de efectos de cada variable (x₁² + x₂² + …), algo que el boosting construye pieza a pieza, mientras que cada árbol del bosque intenta describir la esfera entera con cortes rectos y se queda corto: el bosque arrastra ese **sesgo**. Esto no es una ley: hay datos donde el bosque gana. Pero en datos tabulares, un boosting **bien ajustado** suele ganar por un poco (Lección 6.6).
</details>

---
## 7. 📋 Ficha del modelo

| | AdaBoost | Gradient Boosting |
|---|---|---|
| **Tipo** | Ensamble de tipo **boosting** (clasificación y regresión) | Ensamble de tipo **boosting** (clasificación y regresión) |
| **Intuición** | El estudiante que repasa más lo que falló | El golfista: cada golpe corrige lo que falta |
| **Cómo aprende** | Sube el peso de los ejemplos mal clasificados; cada tocón vota con fuerza según su acierto | Cada árbol pequeño predice los **residuos** (gradiente de la pérdida) y se suma con `learning_rate` |
| **Hiperparámetros clave** | `n_estimators`, `learning_rate` | `learning_rate` + `n_estimators` (con parada temprana), `max_depth`, `subsample` |
| **✅ Fortalezas** | Simple, convierte modelos débiles en fuertes | De los modelos más precisos para datos tabulares · Admite muchas pérdidas |
| **❌ Debilidades** | Sensible al ruido en las etiquetas | Sobreajusta si no se controla · Lento (en fila) · Más perillas que un bosque |
| **Úsalo cuando…** | Quieres entender boosting o tienes datos limpios | Buscas máxima precisión en datos tabulares (en su versión rápida: Lección 6.4) |
| **Evítalo cuando…** | Hay mucho ruido en las etiquetas | No tienes tiempo de ajustar: un Random Forest es más "a prueba de todo" |

## 8. 🧠 Resumen en 5 puntos
1. **Boosting** entrena modelos **en fila**: cada uno se concentra en lo que los anteriores hicieron mal.
2. **AdaBoost** repesa los ejemplos (sube el peso de los errores) y combina tocones con voto ponderado.
3. **Gradient Boosting** entrena cada árbol sobre los **residuos** y suma sus correcciones multiplicadas por `learning_rate`.
4. El boosting reduce el **sesgo**: 400 tocones en fila superan a un árbol enorme. Pero **sí** sobreajusta con demasiados árboles.
5. Perillas clave: `learning_rate` ↔ `n_estimators` (un equipo, mejor con **parada temprana**), árboles **pequeños** (`max_depth` 2–4) y algo de azar (`subsample`).

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈35 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** ¿Qué diferencia **principal** hay entre bagging y boosting?
- a) Bagging usa árboles y boosting no · b) En bagging los modelos se entrenan por separado; en boosting, cada uno depende de los errores del anterior · c) Boosting solo sirve para regresión

**2.** En Gradient Boosting para regresión, ¿qué intenta predecir el árbol número 5?
- a) El valor real · b) Lo que le falta a la predicción acumulada de los 4 anteriores (los residuos) · c) El promedio

**3.** Bajas `learning_rate` de 0.1 a 0.01. Para un resultado parecido, ¿qué debes hacer con `n_estimators`?
- a) Bajarlo · b) Dejarlo igual · c) Subirlo (≈10 veces)

**4.** Tu Gradient Boosting tiene 99.9 % en entrenamiento y la validación empeora a partir del árbol 150 (de 2,000). ¿Qué haces **primero**?
- a) Subir `max_depth` · b) Usar parada temprana o quedarte con ~150 árboles · c) Subir `learning_rate`

**5.** ¿Por qué en boosting se usan árboles **pequeños**?
- a) Porque son más interpretables · b) Porque cada árbol solo debe aportar una corrección modesta; la complejidad la construye la suma · c) Porque scikit-learn no permite árboles grandes

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Paralelo frente a en fila. Los dos suelen usar árboles.
2. **b)** Cada árbol persigue los residuos de lo acumulado hasta ese momento.
3. **c)** Golpes 10 veces más cortos → unas 10 veces más golpes.
4. **b)** El modelo empieza a memorizar después del árbol ~150: ese es el número que te dice la validación. a) y c) empeoran el sobreajuste.
5. **b)** "Aprendices débiles" sumados con cuidado: si cada árbol es fuerte, la suma memoriza enseguida (Ejercicio 9).
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** En la Lección 6.2, `n_estimators` "no se ajustaba": bastaba con poner suficientes árboles. ¿Por qué en boosting **sí** hay que elegirlo con cuidado? Explícalo con la analogía del golfista.

**B. Caso de análisis 💼** Un equipo de fraude con tarjetas tiene 2 millones de transacciones; el 0.2 % son fraudes. Además, se sabe que un 5 % de las etiquetas de "fraude" son errores de captura. Dudan entre AdaBoost y Gradient Boosting con `subsample=0.8` y `min_samples_leaf=50`. ¿Qué les recomendarías y por qué? ¿Qué más les advertirías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** En un bosque, cada árbol es un votante **independiente**: agregar votantes solo estabiliza el promedio. En boosting, cada árbol es un **golpe más** que corrige lo que falta. Cuando la pelota ya está en el hoyo (el patrón real ya está aprendido), los golpes siguientes solo "corrigen" el ruido del césped: el modelo empieza a memorizar. Por eso el número de golpes se elige con validación o parada temprana.

**B.** Mejor **Gradient Boosting** con esa configuración: AdaBoost sube cada vez más el peso de los ejemplos que no logra clasificar, y las etiquetas erróneas son justo ese tipo de ejemplos; acabaría obsesionado con ellas. `subsample` y `min_samples_leaf` altos ayudan a que ningún ejemplo suelto domine. Advertencias:
- Con 2 millones de filas, `GradientBoostingClassifier` será **lento**: mejor `HistGradientBoostingClassifier`, LightGBM o XGBoost (Lección 6.4).
- Con 0.2 % de fraudes, la exactitud no sirve: medir con *recall*, precisión o curva PR (Lecciones 4.2–4.3) y considerar `class_weight` (Lección 2.7).
- Revisar y limpiar las etiquetas erróneas si es posible: ningún modelo arregla del todo una etiqueta mal puesta.
</details>

### 9.3 🧩 Práctica integradora: tercer round en la telefónica

Los 2,000 clientes de telefonía de nuevo. Hasta ahora: árbol ≈0.80 (Lección 5.4) y Random Forest ajustado ≈0.83 (Lección 6.2), en validación cruzada. ¿Puede el boosting superarlos?

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Los datos y el boosting de fábrica**

In [ ]:
rng_tel = np.random.default_rng(7)
n_tel = 2_000
tel = pd.DataFrame({
    "meses_cliente":    rng_tel.integers(1, 72, n_tel),
    "llamadas_soporte": rng_tel.poisson(1.5, n_tel),
    "cargo_mensual":    rng_tel.integers(200, 1_500, n_tel),
    "contrato_anual":   rng_tel.integers(0, 2, n_tel),
})
riesgo = (-2.0 + 1.1 * tel.llamadas_soporte - 0.07 * tel.meses_cliente
          + 0.0025 * tel.cargo_mensual - 2.2 * tel.contrato_anual)
tel["se_fue"] = (rng_tel.random(n_tel) < 1 / (1 + np.exp(-riesgo))).astype(int)

X_tel, y_tel = tel.drop(columns="se_fue"), tel["se_fue"]
X_tel_tr, X_tel_te, y_tel_tr, y_tel_te = train_test_split(X_tel, y_tel, test_size=0.25, stratify=y_tel, random_state=0)

for nombre, m in [("AdaBoost (de fábrica)", AdaBoostClassifier(random_state=0)),
                  ("Gradient Boosting (de fábrica)", GradientBoostingClassifier(random_state=0))]:
    print(f"{nombre:<32} validación cruzada: {cross_val_score(m, X_tel_tr, y_tel_tr, cv=5).mean():.3f}")

✋ ¿Cómo quedan frente al Random Forest ajustado (≈0.830)?

<details><summary><b>👀 Ver respuesta</b></summary>

AdaBoost de fábrica: ≈**0.838**; Gradient Boosting de fábrica: ≈**0.824**. AdaBoost, sin tocar nada, ya supera al bosque ajustado; Gradient Boosting queda un poco por debajo. ¿Por qué el "más sofisticado" pierde? Ve al paso 2.
</details>

**Paso 2 🔮 — ¿Qué tan grandes deben ser los árboles?** AdaBoost usa tocones (`max_depth=1`); Gradient Boosting, por defecto, árboles de profundidad 3. Recuerda cómo se generaron los datos: el riesgo es una **suma** de efectos (llamadas, meses, cargo, contrato), **sin** combinaciones entre variables. Antes de ejecutar: ¿qué `max_depth` crees que le conviene más a Gradient Boosting?

✍️ *Mi apuesta:*

In [ ]:
for d in [1, 2, 3, 5]:     # ✏️
    m = GradientBoostingClassifier(max_depth=d, n_estimators=200, learning_rate=0.1, random_state=0)
    print(f"max_depth={d}: validación cruzada = {cross_val_score(m, X_tel_tr, y_tel_tr, cv=5).mean():.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Gana **`max_depth=1`** (≈0.841) y el resultado empeora al subir la profundidad (≈0.833, ≈0.821 y ≈0.792). Como el riesgo real es una **suma** de efectos de cada variable, los tocones (una variable por árbol) bastan; árboles más profundos buscan combinaciones de variables que **no existen** y terminan aprendiendo ruido. Por eso AdaBoost, con sus tocones, iba bien desde el principio.

En datos reales no conoces la "regla verdadera": por eso `max_depth` se **ajusta** con validación cruzada.
</details>

**Paso 3 ✏️ — El equipo `learning_rate` × `n_estimators`.** Con `max_depth=1`, prueba combinaciones de tasa de aprendizaje y número de árboles.

In [ ]:
TASAS = [0.3, 0.1, 0.03]           # ✏️
N_ARBOLES = [100, 200, 400]        # ✏️

tabla = pd.DataFrame(index=TASAS, columns=N_ARBOLES, dtype=float)
for lr in TASAS:
    for n in N_ARBOLES:
        m = GradientBoostingClassifier(max_depth=1, learning_rate=lr, n_estimators=n, random_state=0)
        tabla.loc[lr, n] = cross_val_score(m, X_tel_tr, y_tel_tr, cv=5).mean()
tabla.index.name, tabla.columns.name = "learning_rate", "n_estimators"
tabla.round(3)

<details><summary><b>👀 Ver respuesta</b></summary>

Lo mejor está arriba a la derecha: `learning_rate=0.3` con 400 árboles (≈**0.848**), y muy cerca `0.3` con 200 (≈0.847). Con `learning_rate=0.03`, ni 400 árboles alcanzan (≈0.828): los golpes son tan cortos que todavía no llegan al hoyo (**subajuste**). Fíjate en la diagonal: bajar la tasa y subir los árboles da resultados parecidos (0.3 × 100 ≈ 0.1 × 400 ≈ 0.845). Es la regla mnemotécnica en acción.
</details>

**Paso 4 ▶️ — Examen final, una sola vez.**

In [ ]:
LR, N = 0.3, 400     # ✏️ pon aquí tu mejor combinación
finalistas = {
    "Árbol (max_depth=5)":           DecisionTreeClassifier(max_depth=5, random_state=0),
    "Random Forest ajustado (6.2)":  RandomForestClassifier(n_estimators=300, min_samples_leaf=5, max_features=1, random_state=0, n_jobs=-1),
    "AdaBoost de fábrica":           AdaBoostClassifier(random_state=0),
    f"Gradient Boosting (tocones, lr={LR}, {N})": GradientBoostingClassifier(max_depth=1, learning_rate=LR, n_estimators=N, random_state=0),
}
for nombre, m in finalistas.items():
    print(f"{nombre:<42} prueba: {m.fit(X_tel_tr, y_tel_tr).score(X_tel_te, y_tel_te):.3f}")

✋ ¿Quién gana en prueba? ¿Es una diferencia grande?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

El Gradient Boosting de tocones queda primero (≈**0.832**), pero **casi empatado** con AdaBoost (≈0.830) y el bosque ajustado (≈0.828); el árbol queda atrás (≈0.810). Con 500 clientes de prueba, 1 punto son 5 clientes: las diferencias entre los tres ensambles están **dentro del ruido**. La prueba confirma lo importante (los ensambles ajustados le ganan al árbol) y para elegir entre ellos nos quedamos con lo que dijo la **validación cruzada**, que usó más datos: Gradient Boosting de tocones.
</details>

**Paso 5 ✋ — Para el gerente.** En dos frases: ¿por qué el modelo ganador no es "más complejo" sino "más adecuado"?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

*"El riesgo de que un cliente se vaya se explica sumando el efecto de unas pocas cosas (llamadas a soporte, antigüedad, contrato y cargo). El modelo ganador suma cientos de reglas sencillas de una sola variable, que imitan justo esa forma; los modelos con reglas más enredadas terminaban aprendiendo casualidades."*
</details>

---
## 📚 Para profundizar
- 📺 **Video:** StatQuest, "AdaBoost, Clearly Explained" y la serie "Gradient Boost Part 1–4" (inglés, con subtítulos).
- 🎮 **Interactivo:** *Gradient Boosting Interactive Playground* (Alex Rogozhnikov): mueve las perillas y mira cómo cambia el modelo.
- 📘 **Documentación:** scikit-learn, guía de usuario, secciones *AdaBoost* y *Gradient Tree Boosting*.

---
⬅️ **Anterior:** [6.2 Bagging y Random Forest: un bosque de árboles distintos](6.2-bagging-y-random-forest.ipynb) · ➡️ **Siguiente:** [6.4 XGBoost, LightGBM y CatBoost: qué cambia entre ellos](6.4-xgboost-lightgbm-y-catboost.ipynb)